In [ ]:
# pip install kaggle pandas scikit-learn matplotlib seaborn

# --- Step 1: Download dataset from Kaggle ---
# You need a Kaggle API key (kaggle.json) placed in ~/.kaggle/
# Get it from: https://www.kaggle.com/settings -> API -> Create New Token
import kaggle
kaggle.api.dataset_download_files(
    'pavansubhasht/ibm-hr-analytics-attrition-dataset',
    path='./data',
    unzip=True
)





In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, RocCurveDisplay)
import matplotlib.pyplot as plt

In [ ]:

# --- Step 2: Load data ---
df = pd.read_csv('./data/WA_Fn-UseC_-HR-Employee-Attrition.csv')


In [ ]:

# --- Step 3: Preprocess ---
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Drop useless constant/ID columns
df = df.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'])

# One-hot encode categorical columns
cat_cols = df.select_dtypes(include='object').columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X = df.drop(columns=['Attrition'])
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:

# --- Step 4: Train ---
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)


In [ ]:
# --- Step 5: Evaluate ---
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

sns_cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", sns_cm)

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("ROC Curve - Attrition Prediction")
plt.show()

In [ ]:


# --- Step 6: Interpret coefficients (odds ratios) ---
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0],
    'Odds_Ratio': np.exp(model.coef_[0])
}).sort_values('Odds_Ratio', ascending=False)

print(coef_df.head(10))   # top features increasing attrition odds
print(coef_df.tail(10))   # top features decreasing attrition odds